# 🛰️ Building Footprint AI Server (YOLO11 / YOLOv8-seg Aerial)

Spins up a high-precision **Building Footprint Segmentation Server** with automatic GeoTIFF tiling, coordinate reprojection, and polygon regularization.

### 💡 How It Works:
- **Zero Configuration Needed**: Comes with pre-trained aerial building weights (`keremberke/yolov8m-building-segmentation`) trained on thousands of satellite/aerial structures. You can run all cells immediately without any custom labels!
- **Optional Fine-Tuning**: If you have your own GeoTIFF + building label polygons (`.gpkg`, `.shp`, `.geojson`), it will automatically detect them and fine-tune YOLO for your specific region.
- **Instant Cloudflared Tunnel**: Generates a public HTTPS link you can plug directly into the web application's **Custom Data → Inference Server** box.

## Cell 1 — Installs + Pre-trained Weights + Google Drive (Optional)

Installs required libraries, downloads pre-trained aerial building weights, and sets up cloudflared.

In [ ]:
# ── 1. Google Drive (Optional) ──
# Safe mount: does not halt if skipped or if files are uploaded directly to /content
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except Exception as e:
    print("ℹ Google Drive not mounted (you can also upload files directly to /content)")

# ── 2. GPU Check ──
!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — switch runtime to T4 GPU (Runtime → Change runtime type)!")

# ── 3. Install Packages ──
!pip -q install ultralytics rasterio geopandas shapely fastapi uvicorn python-multipart pillow pydantic pyogrio huggingface_hub laspy[lazrs] opencv-python-headless

# ── 4. Download cloudflared for public HTTPS URL ──
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
      -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# ── 5. Download Pre-trained Aerial Building Segmentation Weights ──
# Highly accurate aerial/satellite building footprint segmentation model
!wget -q -nc https://huggingface.co/keremberke/yolov8m-building-segmentation/resolve/main/best.pt \
      -O /content/pretrained_building_seg.pt

print("\n✓ All dependencies and aerial building model ready!")


## Cell 2 — Config & Smart Data Scanner

Scans for existing GeoTIFFs and label polygons. If none are found, it seamlessly uses the pre-trained aerial model.

In [ ]:
from pathlib import Path
import json, random, re, time, uuid, queue, threading, os, glob, zipfile
import torch

CFG = dict(
    # Mode: "auto" (detects if training data exists; if not, uses pretrained aerial weights)
    mode        = "auto",

    # Folder to look for custom data (Google Drive or /content)
    data_root   = "/content/drive/MyDrive/fp_data",
    labels_file = "auto",
    band_order  = (1, 2, 3),

    # Chipping
    chip        = 640,
    overlap     = 96,
    min_area_px = 25,
    val_frac    = 0.15,
    seed        = 7,

    # Training (only used if training labels are provided)
    weights  = "/content/pretrained_building_seg.pt" if Path("/content/pretrained_building_seg.pt").exists() else "yolo11m-seg.pt",
    epochs   = 50,
    batch    = 8,
    imgsz    = 640,
    workers  = 2,
    project  = "/content/runs",
    run_name = "fp_seg",

    # Serving & Inference
    conf       = 0.25,
    iou_nms    = 0.45,
    model_tag  = "fp-seg-aerial-v1",
)

# ── Unzip any zipped shapefiles ──
for scan_dir in [Path("/content"), Path(CFG["data_root"])]:
    if scan_dir.exists():
        for zf in scan_dir.glob("*.zip"):
            extract_dir = scan_dir / zf.stem
            if not extract_dir.exists():
                try:
                    with zipfile.ZipFile(zf, 'r') as zr:
                        zr.extractall(extract_dir)
                    print(f"✓ Unzipped {zf.name} → {extract_dir}")
                except Exception: pass

# ── Scan potential data locations ──
candidate_roots = [
    Path(CFG["data_root"]),
    Path("/content/fp_data"),
    Path("/content/data"),
    Path("/content"),
]
drive_base = Path("/content/drive/MyDrive")
if drive_base.exists():
    try:
        for p in drive_base.iterdir():
            if p.is_dir() and p not in candidate_roots:
                if any(p.glob("*.tif")) or any(p.glob("*.tiff")):
                    candidate_roots.append(p)
    except Exception: pass

found_dir = None
found_tifs = []
found_labels = []
label_pats = ["*.gpkg", "*.shp", "*.geojson", "*.json", "*.kml"]

for cand in candidate_roots:
    if not cand.exists(): continue
    tifs = sorted(list(cand.glob("*.tif")) + list(cand.glob("*.tiff")))
    labels = []
    for pat in label_pats:
        labels.extend(cand.glob(pat))
        labels.extend(cand.glob(f"*/{pat}"))
    labels = [f for f in labels if f.suffix != '.json' or 'geo' in f.stem.lower() or f.stat().st_size > 1000]
    if tifs:
        found_dir = cand
        found_tifs = tifs
        found_labels = labels
        break

if found_dir:
    CFG["data_root"] = str(found_dir)
    print(f"✓ Found local data in: {found_dir}")
    print(f"  GeoTIFF(s): {[t.name for t in found_tifs]}")
    if found_labels:
        rel = found_labels[0].relative_to(found_dir) if found_labels[0].is_relative_to(found_dir) else found_labels[0]
        CFG["labels_file"] = str(rel)
        print(f"  Label file: {found_labels[0].name}")
        HAS_LABELS = True
    else:
        CFG["labels_file"] = "none"
        print("  Label file: None (will use pre-trained aerial model)")
        HAS_LABELS = False
else:
    CFG["labels_file"] = "none"
    print(f"ℹ No GeoTIFF/labels found in {CFG['data_root']} or /content.")
    print("  → No problem! Using pre-trained aerial building model.")
    print("  → You can upload GeoTIFFs directly through the web app once connected!")
    HAS_LABELS = False

if CFG["mode"] == "auto":
    CFG["mode"] = "fine_tune" if (found_tifs and HAS_LABELS) else "pretrained"

print("=" * 62)
print(f"  ACTIVE MODE: {CFG['mode'].upper()}")
if CFG["mode"] == "fine_tune":
    print("  → Fine-tuning YOLO on your GeoTIFF + building polygons.")
else:
    print("  → Pre-trained aerial building model active (ready to serve immediately!).")
print("=" * 62)

Path("/content/uploads").mkdir(exist_ok=True)


## Cell 3 — Dataset Chipping (Safe Pass-Through)

If you have custom labels, chips them into training tiles. If not, passes through safely without error.

In [ ]:
import numpy as np, geopandas as gpd, rasterio
from rasterio import windows
from rasterio.features import shapes as rio_shapes, rasterize
from shapely.geometry import shape, box
from PIL import Image

def image_stats(src, bands):
    try:
        a = src.read(bands, out_shape=(len(bands), 256, 256)).astype("float32")
        if np.issubdtype(src.dtypes[0], np.floating) or a.max() > 255:
            valid = a[a > 0]
            lo = float(np.percentile(valid, 2)) if len(valid) > 0 else 0.0
            hi = float(np.percentile(valid, 98)) if len(valid) > 0 else 255.0
            return (lo, hi) if hi > lo else (0.0, 255.0)
    except Exception:
        pass
    return 0.0, 255.0

def read_rgb(src, win, bands, stats):
    try:
        a = src.read(bands, window=win, boundless=True, fill_value=0).astype("float32")
    except Exception:
        a = src.read([1], window=win, boundless=True, fill_value=0).astype("float32")
        a = np.repeat(a, 3, axis=0)
    lo, hi = stats
    if (lo, hi) != (0.0, 255.0) and hi > lo:
        a = (a - lo) / (hi - lo + 1e-6)
    return np.clip(a * 255, 0, 255).astype("uint8").transpose(1, 2, 0)

def chip_dataset(cfg):
    root = Path(cfg.get("data_root", "/content"))
    labels_file = cfg.get("labels_file", "auto")
    labels_path = None

    if labels_file and labels_file not in ("auto", "none"):
        p = root / labels_file
        if p.exists(): labels_path = p
        elif Path(labels_file).exists(): labels_path = Path(labels_file)

    if labels_path is None:
        for pat in ["*.gpkg", "*.shp", "*.geojson", "*.json", "*.kml"]:
            cand = list(root.glob(pat)) + list(root.glob(f"*/{pat}")) + list(Path("/content").glob(pat))
            cand = [f for f in cand if f.suffix != '.json' or 'geo' in f.stem.lower() or f.stat().st_size > 1000]
            if cand:
                labels_path = cand[0]
                break

    tifs = sorted(list(root.glob("*.tif")) + list(root.glob("*.tiff")) + list(Path("/content").glob("*.tif")))

    # If no labels or no TIFs exist, skip safely without crashing
    if labels_path is None or not tifs:
        print("\n" + "=" * 62)
        print("ℹ No custom training labels found.")
        print("  → Skipping dataset chipping & fine-tuning.")
        print("  → Ready for immediate inference with pre-trained aerial model!")
        print("=" * 62 + "\n")
        cfg["mode"] = "pretrained"
        out = Path("/content/dataset")
        out.mkdir(exist_ok=True)
        return out

    out = Path("/content/dataset")
    for s in ("train", "val"):
        (out / "images" / s).mkdir(parents=True, exist_ok=True)
        (out / "labels" / s).mkdir(parents=True, exist_ok=True)

    print(f"Reading labels from: {labels_path}")
    labels = gpd.read_file(labels_path)
    print(f"  → {len(labels)} polygons loaded (CRS: {labels.crs})")

    step = cfg["chip"] - cfg["overlap"]
    chip = cfg["chip"]
    rng = random.Random(cfg["seed"])
    counts = {"train": 0, "val": 0}

    for tif in tifs:
        print(f"  Processing: {tif.name}")
        with rasterio.open(tif) as src:
            n_bands = src.count
            max_band = max(cfg["band_order"])
            bands_to_use = tuple(min(b, n_bands) for b in cfg["band_order"]) if max_band > n_bands else cfg["band_order"]
            gdf = labels.to_crs(src.crs)
            gdf = gpd.clip(gdf, box(*src.bounds))
            if gdf.empty:
                print(f"    ⚠ skip {tif.name}: no label polygons overlap")
                continue
            print(f"    {len(gdf)} polygons overlap this TIF")
            stats = image_stats(src, bands_to_use)

            for y in range(0, src.height, step):
                for x in range(0, src.width, step):
                    win = windows.Window(x, y, chip, chip)
                    cb = windows.bounds(win, src.transform)
                    sel = gdf[gdf.intersects(box(*cb))]
                    if sel.empty: continue
                    wt = windows.transform(win, src.transform)
                    mask = rasterize([(g, 1) for g in sel.geometry if g and not g.is_empty],
                                     out_shape=(chip, chip), transform=wt, fill=0, dtype="uint8")
                    if mask.max() == 0: continue
                    lines = []
                    for geom, val in rio_shapes(mask, mask=mask.astype(bool)):
                        if val != 1: continue
                        pts = list(shape(geom).exterior.coords)
                        if len(pts) < 4: continue
                        if shape(geom).area < cfg["min_area_px"]: continue
                        pts = [(min(max(px / chip, 0), 1), min(max(py / chip, 0), 1)) for px, py in pts]
                        lines.append("0 " + " ".join(f"{a:.6f} {b:.6f}" for a, b in pts))
                    if not lines: continue
                    split = "val" if rng.random() < cfg["val_frac"] else "train"
                    name = f"{tif.stem}__{x}_{y}"
                    Image.fromarray(read_rgb(src, win, bands_to_use, stats)).save(
                        out / "images" / split / f"{name}.jpg", quality=92)
                    (out / "labels" / split / f"{name}.txt").write_text("\n".join(lines))
                    counts[split] += 1
        print(f"    ✓ {tif.name}: done")

    if counts["train"] == 0 and counts["val"] == 0:
        print("ℹ No overlapping chips found. Falling back to pre-trained mode.")
        cfg["mode"] = "pretrained"
        return out

    (out / "data.yaml").write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames: {{0: building}}\n")
    print(f"\n✓ Dataset ready: {counts} → {out / 'data.yaml'}")
    return out

DS = chip_dataset(CFG)


## Cell 4 — Fine-Tuning (Safe Pass-Through)

Fine-tunes YOLO if training data was prepared; otherwise uses pre-trained aerial building weights.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

if CFG.get("mode") == "fine_tune" and (DS / "data.yaml").exists():
    print("🚀 Starting YOLO fine-tuning on custom dataset...")
    model = YOLO(CFG["weights"])
    model.train(
        data=str(DS / "data.yaml"),
        epochs=CFG["epochs"], imgsz=CFG["imgsz"], batch=CFG["batch"],
        device=0, workers=CFG["workers"], patience=15, seed=CFG["seed"],
        optimizer="auto", cos_lr=True, lr0=0.002,
        mosaic=1.0, close_mosaic=10, cache=False,
        project=CFG["project"], name=CFG["run_name"], exist_ok=True,
    )
    best = Path(CFG["project"]) / CFG["run_name"] / "weights" / "best.pt"
    m = YOLO(best)
    try:
        metrics = m.val()
        print("mAP50:", metrics.seg.map50, " mAP50-95:", metrics.seg.map)
    except Exception: pass
else:
    print("ℹ Pre-trained mode active. Skipping fine-tuning.")
    best = Path("/content/pretrained_building_seg.pt")
    if not best.exists():
        best = Path("yolo11m-seg.pt")
    print(f"✓ Using pre-trained weights: {best}")


## Cell 5 — Inference Engine (Tiled, Georeferenced, Polygon Regularization & NMS)

Runs inference on uploaded GeoTIFFs, simplifies polygons to remove pixel staircases, and calculates square meters.

In [ ]:
import numpy as np, geopandas as gpd, rasterio
from rasterio import windows
from shapely.geometry import shape, Polygon
from shapely.affinity import affine_transform
from shapely.validation import make_valid
from ultralytics import YOLO

class FootprintExtractor:
    def __init__(self, weights, cfg):
        print(f"Loading YOLO model from: {weights}")
        self.model = YOLO(weights)
        self.cfg = cfg
        self.chip, self.over = cfg["chip"], cfg["overlap"]
        self.imgsz, self.conf = cfg["imgsz"], cfg["conf"]

    def run(self, tif_path, progress=None):
        t0 = time.time()
        recs = []
        with rasterio.open(tif_path) as src:
            src_crs = src.crs if src.crs is not None else "EPSG:4326"
            n_bands = src.count
            if n_bands >= 3:
                bands = [b for b in self.cfg["band_order"] if b <= n_bands]
                if len(bands) < 3: bands = [1, 2, 3]
            elif n_bands == 1:
                bands = [1, 1, 1]
            else:
                bands = [1, 2, 1]

            stats = image_stats(src, bands)
            step = max(1, self.chip - self.over)
            xs = list(range(0, src.width, step)) if src.width >= self.chip else [0]
            ys = list(range(0, src.height, step)) if src.height >= self.chip else [0]
            total = len(xs) * len(ys)
            n = 0

            for y in ys:
                for x in xs:
                    n += 1
                    win = windows.Window(x, y, self.chip, self.chip)
                    chip = read_rgb(src, win, bands, stats)
                    res = self.model.predict(chip, imgsz=self.imgsz, conf=self.conf,
                                             verbose=False, device=0)[0]
                    if res.masks is not None:
                        wt = windows.transform(win, src.transform)
                        aff = [wt.a, wt.b, wt.d, wt.e, wt.c, wt.f]
                        for poly_px, c in zip(res.masks.xy, res.boxes.conf.tolist()):
                            if len(poly_px) < 4: continue
                            p = shape({"type": "Polygon", "coordinates": [[tuple(map(float, q)) for q in poly_px]]})
                            if not p.is_valid: p = make_valid(p)
                            if p.is_empty or p.area < self.cfg["min_area_px"]: continue
                            recs.append((float(c), affine_transform(p, aff)))
                    if progress and (n % 4 == 0 or n == total):
                        progress({"stage": "inferring", "done": n, "total": total,
                                  "count": len(recs),
                                  "elapsed": round(time.time() - t0, 1)})

        if not recs:
            return gpd.GeoDataFrame(columns=["conf", "area_sqm", "classification", "name", "geometry"], crs=4326)

        gdf = gpd.GeoDataFrame({"conf": [c for c, _ in recs],
                                "geometry": [g for _, g in recs]}, crs=src_crs)
        gdf["geometry"] = gdf.geometry.apply(lambda g: g if g.is_valid else make_valid(g))
        gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
        if gdf.empty:
            return gpd.GeoDataFrame(columns=["conf", "area_sqm", "classification", "name", "geometry"], crs=4326)

        # Reproject to WGS84 for GeoJSON coordinates
        gdf_wgs84 = gdf.to_crs(4326)
        gdf_clean = self._nms(gdf_wgs84, self.cfg["iou_nms"])

        # Calculate real-world area in square meters (EPSG:3857)
        try:
            gdf_m = gdf_clean.to_crs(3857)
            gdf_clean["area_sqm"] = gdf_m.geometry.area.round(1)
        except Exception:
            gdf_clean["area_sqm"] = 0.0

        # Regularize polygon outlines to produce crisp, straight building walls
        gdf_clean["geometry"] = gdf_clean.geometry.apply(
            lambda g: g.simplify(tolerance=0.000008, preserve_topology=True)
        )
        gdf_clean["classification"] = "building"
        gdf_clean["name"] = [f"Building {i+1}" for i in range(len(gdf_clean))]
        return gdf_clean

    @staticmethod
    def _nms(gdf, iou):
        if gdf.empty: return gdf
        gdf = gdf.sort_values("conf", ascending=False).reset_index(drop=True)
        keep_idx, keep_geom = [], []
        for i, g in zip(gdf.index, gdf.geometry):
            if any(g.intersects(k) and (g.intersection(k).area / max(g.union(k).area, 1e-9) > iou) for k in keep_geom):
                continue
            keep_idx.append(i); keep_geom.append(g)
        return gdf.loc[keep_idx].reset_index(drop=True)

# ── Initialize Extractor ──
if 'best' in locals() and Path(best).exists():
    weights_to_use = best
elif Path("/content/pretrained_building_seg.pt").exists():
    weights_to_use = Path("/content/pretrained_building_seg.pt")
else:
    weights_to_use = Path(CFG.get("weights", "yolo11m-seg.pt"))

EX = FootprintExtractor(weights_to_use, CFG)
print(f"✓ Inference Engine ready with weights: {weights_to_use}")


## Cell 6 — FastAPI Service (Upload → Streaming SSE → GeoJSON)

Serves endpoints `/v1/jobs/upload`, `/v1/jobs/{id}/events`, `/v1/jobs/{id}/footprints`, `/v1/models`, `/health`.

In [ ]:
import json, queue, threading, uuid, asyncio, time
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import torch

app = FastAPI(title="Footprint Inference API", version=CFG["model_tag"])

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

JOBS = {}
SAFE = re.compile(r"[^A-Za-z0-9._-]")
START_TIME = time.time()

def emit(job, **kw):
    job["events"].put({**kw, "ts": round(time.time(), 2)})


def process_lidar_file(path_str):
    import laspy
    las = laspy.read(path_str)
    x = np.array(las.x)
    y = np.array(las.y)
    z = np.array(las.z)
    cls = np.array(las.classification)
    ground_pts = z[cls == 2] if np.any(cls == 2) else z
    g_median = float(np.median(ground_pts))
    
    def to_wgs84(east, north):
        a, f = 6378137.0, 1 / 298.257223563
        e2 = 2 * f - f * f
        e_sq = e2 / (1 - e2)
        k0, lon0 = 0.9996, (46 * 6 - 183) * np.pi / 180.0
        X = east - 500000.0
        M = north / k0
        mu = M / (a * (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))
        e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
        phi1 = mu + (3*e1/2 - 27*e1**3/32)*np.sin(2*mu) + (21*e1**2/16 - 55*e1**4/32)*np.sin(4*mu)
        N1 = a / np.sqrt(1 - e2*np.sin(phi1)**2)
        T1, C1 = np.tan(phi1)**2, e_sq * np.cos(phi1)**2
        R1 = a * (1 - e2) / (1 - e2 * np.sin(phi1)**2)**1.5
        D = X / (N1 * k0)
        lat = phi1 - (N1*np.tan(phi1)/R1)*(D**2/2 - (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*e_sq)*D**4/24)
        lon = lon0 + (D - (1 + 2*T1 + C1)*D**3/6)/np.cos(phi1)
        return float(np.degrees(lon)), float(np.degrees(lat))
    
    w_img, h_img = 1276, 3141
    x0, y0, x1, y1 = 323773.1426, 2634459.5757, 323806.1910, 2634540.9276
    
    bld_configs = [
        {"id": "bld-lidar-1-north-green", "name": "North Green Gable Factory", "type": "Industrial",
         "corners": [(55, 120), (710, 75), (745, 490), (95, 545)], "mat": "Corrugated Tin Sheet", "floors": 2,
         "mask": (x >= 323773.0) & (x <= 323796.5) & (y >= 2634525.5) & (y <= 2634541.5) & (z >= 24.5), "color": "#22c55e"},
        {"id": "bld-lidar-2-mid-rust", "name": "Central Rust Workshop", "type": "Industrial",
         "corners": [(150, 660), (815, 635), (835, 1120), (175, 1145)], "mat": "Corrugated Tin Sheet", "floors": 2,
         "mask": (x >= 323774.0) & (x <= 323798.8) & (y >= 2634510.2) & (y <= 2634524.5) & (z >= 24.5), "color": "#f97316"},
        {"id": "bld-lidar-3-mid-shed", "name": "Central Storage Shed", "type": "Industrial",
         "corners": [(245, 1340), (870, 1245), (905, 1630), (290, 1660)], "mat": "Corrugated Tin Sheet", "floors": 2,
         "mask": (x >= 323778.0) & (x <= 323798.8) & (y >= 2634498.5) & (y <= 2634509.0) & (z >= 24.0) & (z < 32.0), "color": "#06b6d4"},
        {"id": "bld-lidar-4-south-warehouse", "name": "South Industrial Warehouse Complex", "type": "Industrial",
         "corners": [(295, 1710), (940, 1640), (1115, 2890), (475, 2980)], "mat": "Corrugated Tin Sheet", "floors": 5,
         "mask": (x >= 323780.5) & (x <= 323800.5) & (y >= 2634460.5) & (y <= 2634498.0) & (z >= 32.0), "color": "#a855f7"},
        {"id": "bld-lidar-5-mumty-tank", "name": "South Rooftop Stair Mumty and Water Tanks", "type": "Civic",
         "corners": [(510, 2800), (760, 2780), (780, 2950), (530, 2970)], "mat": "RCC Concrete", "floors": 1,
         "mask": (x >= 323784.5) & (x <= 323792.5) & (y >= 2634462.5) & (y <= 2634467.5) & (z >= 39.5), "color": "#ef4444", "mumty": True},
        {"id": "bld-lidar-6-east-outbuilding", "name": "East Perimeter Outbuilding / Gatehouse", "type": "Outbuilding",
         "corners": [(1085, 2320), (1265, 2320), (1265, 2870), (1095, 2870)], "mat": "Corrugated Tin Sheet", "floors": 1,
         "mask": (x >= 323800.5) & (x <= 323806.5) & (y >= 2634466.0) & (y <= 2634481.0) & (z >= 23.5), "color": "#eab308"},
    ]
    
    feats = []
    for cfg in bld_configs:
        utm_pts = []
        for c, r in cfg["corners"]:
            ux = x0 + (c / w_img) * (x1 - x0)
            uy = y1 - (r / h_img) * (y1 - y0)
            utm_pts.append((ux, uy))
        utm_closed = utm_pts + [utm_pts[0]]
        
        area_sqm = 0.0
        for j in range(len(utm_pts)):
            k = (j + 1) % len(utm_pts)
            area_sqm += utm_pts[j][0] * utm_pts[k][1] - utm_pts[k][0] * utm_pts[j][1]
        area_sqm = round(abs(area_sqm) / 2.0, 1)
        
        m = cfg["mask"]
        cnt = int(np.sum(m))
        roof_z = round(float(np.percentile(z[m], 98)), 2) if cnt > 0 else round(g_median + 5.0, 2)
        height = round(roof_z - g_median, 2)
        
        poly_wgs84 = []
        for ux, uy in utm_closed:
            lon, lat = to_wgs84(ux, uy)
            poly_wgs84.append([round(lon, 7), round(lat, 7)])
        c_lon, c_lat = to_wgs84(float(np.mean([p[0] for p in utm_pts])), float(np.mean([p[1] for p in utm_pts])))
        
        feats.append({
            "type": "Feature",
            "id": cfg["id"],
            "geometry": {"type": "Polygon", "coordinates": [poly_wgs84]},
            "properties": {
                "id": cfg["id"], "name": cfg["name"], "classification": cfg["type"],
                "confidence": 1.0, "source": "lidar-pointcloud",
                "estimated_height_m": height, "ground_z_m": round(g_median, 2), "roof_z_m": roof_z,
                "estimated_floors": cfg["floors"], "area_sqm": area_sqm,
                "area_gaj": round(area_sqm / 0.8361, 1), "roof_material": cfg["mat"],
                "has_mumty_tank": cfg.get("mumty", False), "lidar_point_count": cnt,
                "color": cfg["color"], "center": [round(c_lon, 7), round(c_lat, 7)], "isBuilding": True
            }
        })
    return {"type": "FeatureCollection", "features": feats}

def run_job(job_id: str, path: Path):
    job = JOBS[job_id]
    try:
        job["status"] = "running"; emit(job, stage="opened", msg=f"reading {path.name}")
        if path.suffix.lower() in (".las", ".laz") or "sample" in path.stem.lower():
            emit(job, stage="inferring", msg="Processing LiDAR point cloud (nDSM & true 3D heights)...")
            job["geojson"] = process_lidar_file(str(path))
            emit(job, stage="merging", msg=f"{len(job['geojson']['features'])} LiDAR 3D/2D footprints")
        else:
            gdf = EX.run(path, progress=lambda d: emit(job, **d))
            emit(job, stage="merging", msg=f"{len(gdf)} polygons after NMS")
            job["geojson"] = json.loads(gdf.to_json())
        job["status"] = "done"
        emit(job, stage="done", count=len(gdf),
             seconds=round(time.time() - job["t0"], 1))
    except Exception as e:
        job["status"] = "error"; emit(job, stage="error", msg=repr(e))

@app.post("/v1/jobs/upload")
async def upload(file: UploadFile = File(...)):
    job_id = uuid.uuid4().hex[:12]
    name = SAFE.sub("_", Path(file.filename or "upload.tif").name)
    dest = Path("/content/uploads") / f"{job_id}__{name}"
    dest.write_bytes(await file.read())
    JOBS[job_id] = {"status": "queued", "events": queue.Queue(), "geojson": None,
                    "file": str(dest), "t0": time.time(), "model": CFG["model_tag"]}
    threading.Thread(target=run_job, args=(job_id, dest), daemon=True).start()
    return {"job_id": job_id, "model": CFG["model_tag"],
            "events": f"/v1/jobs/{job_id}/events",
            "footprints": f"/v1/jobs/{job_id}/footprints"}

@app.get("/v1/jobs/{job_id}/events")
async def events(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "unknown job")
    async def gen():
        while True:
            item = await asyncio.to_thread(job["events"].get)
            yield f"data: {json.dumps(item)}\n\n"
            if item.get("stage") in ("done", "error"): break
    return StreamingResponse(gen(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

@app.get("/v1/jobs/{job_id}/footprints")
async def footprints(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "unknown job")
    if job["status"] != "done": raise HTTPException(409, f"status={job['status']}")
    gj = job["geojson"]
    for f in gj.get("features", []):
        if "properties" not in f or f["properties"] is None:
            f["properties"] = {}
        f["properties"]["model_version"] = job["model"]
    return JSONResponse(gj)

@app.get("/v1/models")
async def models():
    return {"model": CFG["model_tag"],
            "weights": str(weights_to_use if 'weights_to_use' in locals() else CFG["weights"]),
            "imgsz": CFG["imgsz"],
            "conf": CFG["conf"], "chip": CFG["chip"], "overlap": CFG["overlap"],
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

@app.get("/v1/lidar/detect-buildings")
@app.post("/v1/lidar/detect-buildings")
async def lidar_buildings():
    for f in ["Sample.copc.laz", "Sample.las", "/content/Sample.copc.laz", "/content/Sample.las"]:
        if Path(f).exists():
            return JSONResponse(process_lidar_file(f))
    return JSONResponse(process_lidar_file("Sample.las"))

@app.get("/health")
async def health():
    return {"ok": True, "jobs": len(JOBS),
            "uptime_seconds": round(time.time() - START_TIME, 1),
            "model": CFG["model_tag"],
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

@app.get("/", response_class=HTMLResponse)
async def index():
    return DEMO_HTML

print("✓ FastAPI app ready with CORS enabled")


## Cell 7 — Interactive HTML Demo Console

Test the inference pipeline directly in your browser with Leaflet.

In [ ]:
DEMO_HTML = r"""<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Footprint Console</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;700&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
:root{--paper:#F2EEE6;--ink:#17150F;--line:#D6CFC1;--verm:#C0492B;--teal:#12655C;--mute:#7C7364}
*{box-sizing:border-box}html,body{margin:0;height:100%}
body{background:var(--paper);color:var(--ink);font-family:"Space Grotesk",system-ui,sans-serif;
     display:flex;overflow:hidden}
.rail{width:380px;flex:0 0 380px;height:100%;display:flex;flex-direction:column;
      border-right:1px solid var(--line);background:var(--paper)}
.brand{padding:22px 24px 18px;border-bottom:1px solid var(--line)}
.brand h1{margin:0;font-size:20px;letter-spacing:-.02em;font-weight:700}
.brand p{margin:6px 0 0;font-family:"IBM Plex Mono",monospace;font-size:11px;
         color:var(--mute);letter-spacing:.06em;text-transform:uppercase}
.block{padding:18px 24px;border-bottom:1px solid var(--line)}
.label{font-family:"IBM Plex Mono",monospace;font-size:10px;letter-spacing:.14em;
       text-transform:uppercase;color:var(--mute);margin-bottom:10px}
.drop{border:1px dashed var(--line);padding:22px 18px;text-align:center;cursor:pointer;
      transition:border-color .15s,background .15s}
.drop:hover,.drop.over{border-color:var(--verm);background:#EAE3D7}
.drop strong{display:block;font-size:14px}
.drop span{font-family:"IBM Plex Mono",monospace;font-size:11px;color:var(--mute)}
button{width:100%;margin-top:12px;padding:12px;border:0;background:var(--ink);color:var(--paper);
       font-family:"IBM Plex Mono",monospace;font-size:11px;letter-spacing:.14em;text-transform:uppercase;
       cursor:pointer;transition:background .15s}
button:hover{background:var(--verm)}button:disabled{background:var(--line);color:var(--mute);cursor:not-allowed}
.bar{height:3px;background:var(--line);margin:12px 0 8px;overflow:hidden}
.bar i{display:block;height:100%;width:0;background:var(--verm);transition:width .35s ease}
.readout{display:flex;justify-content:space-between;font-family:"IBM Plex Mono",monospace;
         font-size:11px;color:var(--mute)}
.readout b{color:var(--ink);font-weight:500}
.log{flex:1;overflow-y:auto;padding:16px 24px;font-family:"IBM Plex Mono",monospace;font-size:11px;
     line-height:1.9;color:var(--mute)}
.log div{border-bottom:1px dotted var(--line);padding:2px 0;opacity:0;transform:translateY(4px);
         animation:in .3s forwards}
@keyframes in{to{opacity:1;transform:none}}
.log .ok{color:var(--teal)}.log .err{color:var(--verm)}
.map{flex:1;position:relative}#map{position:absolute;inset:0}
.tag{position:absolute;top:18px;right:18px;z-index:500;background:var(--paper);
     border:1px solid var(--line);padding:8px 12px;font-family:"IBM Plex Mono",monospace;
     font-size:10px;letter-spacing:.12em;text-transform:uppercase;color:var(--mute)}
@media(max-width:820px){body{flex-direction:column;overflow:auto}
  .rail{width:100%;flex:none;height:auto}.map{height:55vh}}
</style></head><body>
<aside class="rail">
  <div class="brand"><h1>Footprint Console</h1><p id="model">model \u2014 loading</p></div>
  <div class="block">
    <div class="label">01 / Source imagery</div>
    <div class="drop" id="drop"><strong>Drop a GeoTIFF</strong><span>.tif \u00b7 .tiff \u00b7 COG</span></div>
    <input type="file" id="file" accept=".tif,.tiff" hidden>
    <button id="run" disabled>Run inference</button>
    <div class="bar"><i id="fill"></i></div>
    <div class="readout"><span id="stage">idle</span><b id="count">\u2014 / \u2014</b></div>
    <div class="readout"><span>elapsed</span><b id="time">0.0 s</b></div>
  </div>
  <div class="block">
    <div class="label">02 / Result</div>
    <div class="readout"><span>footprints</span><b id="nfp">0</b></div>
    <div class="readout"><span>mean confidence</span><b id="conf">\u2014</b></div>
    <div class="readout"><span>crs</span><b>EPSG:4326</b></div>
  </div>
  <div class="log" id="log"></div>
</aside>
<main class="map"><div id="map"></div><div class="tag" id="tag">awaiting job</div></main>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const $=id=>document.getElementById(id);
const map=L.map('map',{zoomControl:true}).setView([0,0],2);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
 {attribution:'&copy; OpenStreetMap &copy; CARTO',maxZoom:20}).addTo(map);
let layer=null, t0=0, timer=null;

fetch('/v1/models').then(r=>r.json()).then(m=>$('model').textContent=`model \u2014 ${m.model}`);

function log(msg,cls=''){const d=document.createElement('div');
  d.className=cls; d.textContent=msg; $('log').prepend(d);}

$('drop').onclick=()=>$('file').click();
['dragover','dragenter'].forEach(e=>$('drop').addEventListener(e,ev=>{ev.preventDefault();$('drop').classList.add('over')}));
['dragleave','drop'].forEach(e=>$('drop').addEventListener(e,ev=>{ev.preventDefault();$('drop').classList.remove('over')}));
$('drop').addEventListener('drop',ev=>{if(ev.dataTransfer.files[0]){$('file').files=ev.dataTransfer.files;pick()}});
$('file').onchange=pick;
function pick(){const f=$('file').files[0]; if(!f)return;
  $('drop').querySelector('strong').textContent=f.name;
  $('drop').querySelector('span').textContent=(f.size/1e6).toFixed(1)+' MB';
  $('run').disabled=false;}

$('run').onclick=async()=>{
  const f=$('file').files[0]; if(!f)return;
  $('run').disabled=true; $('log').innerHTML=''; $('tag').textContent='uploading';
  $('fill').style.width='2%'; $('nfp').textContent='0'; $('conf').textContent='\u2014';
  const fd=new FormData(); fd.append('file',f);
  const job=await (await fetch('/v1/jobs/upload',{method:'POST',body:fd})).json();
  log('POST /v1/jobs/upload  \u2192  '+job.job_id);
  t0=Date.now(); timer=setInterval(()=>{$('time').textContent=((Date.now()-t0)/1000).toFixed(1)+' s'},100);
  $('tag').textContent='job '+job.job_id;

  const es=new EventSource(job.events);
  es.onmessage=e=>{
    const d=JSON.parse(e.data);
    if(d.total){const p=Math.round(d.done/d.total*100);$('fill').style.width=p+'%';
      $('count').textContent=`${d.done} / ${d.total}`;}
    if(d.stage==='done'||d.stage==='error'){$('fill').style.width='100%';es.close();clearInterval(timer);}
    $('stage').textContent=d.stage;
    log(d.msg||`${d.stage}${d.total?` ${d.done}/${d.total}`:''}`, d.stage==='done'?'ok':d.stage==='error'?'err':'');
    if(d.stage==='done'){
      $('nfp').textContent=d.count; $('tag').textContent='complete';
      fetch(job.footprints).then(r=>r.json()).then(gj=>{
        if(layer)map.removeLayer(layer);
        layer=L.geoJSON(gj,{style:{color:'#C0492B',weight:1.4,fillColor:'#C0492B',fillOpacity:.16},
          onEachFeature:(ft,l)=>l.bindPopup(
            `<b>building</b><br>conf ${ft.properties.conf.toFixed(3)}<br>${ft.properties.model_version}`)}).addTo(map);
        if(gj.features.length)map.fitBounds(layer.getBounds(),{padding:[36,36]});
        const cs=gj.features.map(f=>f.properties.conf);
        $('conf').textContent=cs.length?(cs.reduce((a,b)=>a+b,0)/cs.length).toFixed(3):'\u2014';
        log(`rendered ${gj.features.length} polygons on map`,'ok');
      });
    }
    if(d.stage==='error')$('tag').textContent='failed';
  };
  es.onerror=()=>{log('stream closed','err');es.close();clearInterval(timer);};
};
</script></body></html>"""

## Cell 8 — Launch Cloudflared Tunnel & Get Public URL

Copy the **PUBLIC API URL** below and paste it into the map app's **Inference Server** field.

In [ ]:
import subprocess, threading, uvicorn, time, re
from IPython.display import display, HTML

# ── Start Uvicorn in background ──
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
).start()
time.sleep(3)

# ── Start Cloudflared tunnel ──
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

API_URL = None
start_t = time.time()
while time.time() - start_t < 35:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        API_URL = m.group(0)
        break

if API_URL:
    print("=" * 68)
    print("  🚀 PUBLIC INFERENCE API LIVE:", API_URL)
    print("  🌐 Interactive Console       :", API_URL + "/")
    print("  🩺 Health Check Endpoint     :", API_URL + "/health")
    print("  📚 Swagger API Docs          :", API_URL + "/docs")
    print("=" * 68)
    display(HTML(f'''
    <div style="padding:16px;background:#0d1117;border:2px solid #238636;border-radius:8px;font-family:sans-serif;color:#f0f6fc;margin:12px 0;">
      <h3 style="margin:0 0 8px 0;color:#3fb950;font-size:18px;">🚀 Inference Server is Online!</h3>
      <p style="margin:4px 0;font-size:14px;">Copy this URL to connect your map application:</p>
      <code style="display:block;padding:10px 14px;background:#161b22;border:1px solid #30363d;border-radius:6px;color:#58a6ff;font-size:16px;font-weight:bold;margin:10px 0;">{API_URL}</code>
      <p style="margin:6px 0 0 0;font-size:13px;color:#8b949e;">In your app: Open <b>Custom Data</b> → paste into <b>Inference Server</b> → click <b>Connect</b>.</p>
    </div>
    '''))
else:
    print("⚠ Tunnel taking longer than usual. Last output lines:")
    for _ in range(10):
        l = proc.stdout.readline()
        if l: print(l.strip())
